# A Real Tool-Calling Loop: `make_react_agent`

https://docs.langchain.com/oss/python/langchain/agents defines an agent as
"a model calling tools in a loop until a given task is complete." psychscanner's
built-in agent (see the previous notebook, section 1) binds tools with
`model.bind_tools(...)` but never actually **executes** a requested tool and
loops on the result — see `memories.single_turn_convo.call_model`. That real
loop doesn't exist in psychscanner, so `psychscanner.agents.make_react_agent`
wraps LangChain's own `create_agent` (the API on the page above) as a
`ScanningAgent`, the same "bring your own agent" seam
`psychscanner.agents.CustomAgent` uses.

This notebook plugs it in two ways:
1. Directly through `TaskRunner` (no `ExpCard` needed)
2. Through `ScannerModel.run(custom_agent=...)` for a full simulation run

Needs a tool-calling-capable model — `mock-llm` doesn't implement `bind_tools`.
Uses the local `llama3.2:3b` Ollama model pulled for this tutorial series.

In [1]:
from pathlib import Path

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

import psychscanner as psy
from psychscanner.agents import make_react_agent
from psychscanner.memories import llm_chat_model
from psychscanner.task_runner import TaskRunner

print("PsychScanner successfully imported!")

PsychScanner successfully imported!


In [2]:
@tool
def web_search(query: str) -> str:
    """Search the web for a query."""
    facts = {
        "vviq": "The VVIQ (Vividness of Visual Imagery Questionnaire) is a 16-item self-report measure of imagery vividness.",
    }
    match = next((v for k, v in facts.items() if k in query.lower()), None)
    return match or f"[no canned result for '{query}']"


model = llm_chat_model(model="llama3.2:3b", family="ollama", parameters={"temperature": 0})
agent = make_react_agent(
    model, [web_search],
    system_message="You are a helpful research assistant. Always call web_search with the term 'vviq' "
                    "before answering a question about it, then answer using only what the tool returned.",
)
print(agent.ai_app is agent)  # CustomAgent wraps itself as its own ai_app, same contract as before

--<api key>-- warning: OLLAMA_API_KEY not set; proceeding without explicit api_key for family 'ollama'


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model='llama3.2:3b' temperature=0.0


True


## 1. Directly through `TaskRunner`

In [3]:
tasktrials = {"trials": [
    {"trcode": "t1", "stimulus": HumanMessage(content="What does 'vviq' measure? Use web_search to check, then answer in one sentence."),
     "tasktype": "x", "parser": None, "fb": False},
]}
runner = TaskRunner(
    scanning_agent=agent, trace_cfg={"trial": "tut-", "task": "tut-task"},
    system_message="You are a helpful research assistant.",
    tasktrials=tasktrials, chain_type="item", hmsg="stimulus",
)
for r in runner.execute():
    print(r["trcode"], "->", r["pred_resp"].content)

----<>---- task running


t1 -> The VVIQ measures the vividness of visual imagery, which refers to how clear and detailed one's mental images are.


## 2. Through `ScannerModel.run(custom_agent=...)`

Same as the custom-agent tutorial: `custom_agent` bypasses `AgentConfig`'s
LangChain model entirely, so the card's `model`/`family` fields don't matter
here — the default `mock-llm` is fine. `chain_type`/`memory` still control
how many system-message "runs" `ScannerModel` loops over and what
`thread_id` gets passed in `config`, but `make_react_agent`'s graph has no
checkpointer, so it doesn't use it — every trial is independent regardless
of `chain_type`, same as `SingleTurn`. Note also that the task JSON's
`instructions` become the *task's* system message, but `make_react_agent`'s
`call_fn` only reads `input_dict["inputs"]` — the agent's own system message
is the one fixed in the `make_react_agent(...)` call above, same as any
`CustomAgent`.

In [4]:
RUN_DIR = Path.cwd() / "_react_agent_tutorial_run"

task = {
    "tasktype": "survey", "taskname": "react_agent_demo",
    "instructions": {"definition": ["Always call the web_search tool first to fact-check, then answer in one sentence."]},
    "contexts": ["General knowledge"], "contexts_id": ["gk"], "context_present": False,
    "chain_type": "item", "parser": "0",
    "items": {"gk": [
        {"trcode": "gk_1", "stimulus": "What does 'vviq' measure? Use web_search to check, then answer in one sentence."},
    ]},
}

card_in = psy.ExpCardInit()
card_in.proj_dir, card_in.projectname = RUN_DIR, "react_agent_demo"
card_in.task_file = task
card_in.tunnel_status = "0"
card_in.cogtype, card_in.nsim = "no", 1
card_in.chain_type, card_in.memory = "item", "SingleTurn"

expcard = psy.ExpCard(card_in)
scanner = psy.ScannerModel(expcard=expcard)
scan_results = scanner.run(custom_agent=agent)
for trial in scan_results[0]:
    print(trial["trcode"], "->", trial["pred_resp"].content)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_react_agent_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_react_agent_tutorial_run/react_agent_demo/react_agent_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

1it [00:17, 17.27s/it]


2026-07-06 11:48:31.872 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:48:31.875 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


gk_1 -> The VVIQ measures the vividness of visual imagery, which refers to how clear and detailed one's mental images are.


## Recap

- An agent is "a model calling tools in a loop until complete" — LangChain's
  own `create_agent` already implements that loop; `make_react_agent` just
  adapts it to psychscanner's `ScanningAgent` contract instead of
  reimplementing the loop.
- The built-in agent (previous notebook) *binds* tools but doesn't execute
  them; this one actually runs the tool and continues, exactly like the
  agent docs describe.
- Plug it in the same two ways any `ScanningAgent` plugs in:
  `TaskRunner(scanning_agent=...)` directly, or `ScannerModel.run(custom_agent=...)`
  for a full simulation.

---
## Further reading

Advanced applications of the reason-act-observe tool-calling loop `make_react_agent` wraps:

1. **["ReAct: Synergizing Reasoning and Acting in Language Models"](https://arxiv.org/abs/2210.03629)** (Yao et al., 2023) — the paper this pattern is named after: interleaving reasoning traces with tool calls so each informs the next, exactly the loop `create_agent` executes under the hood here.
2. **["Toolformer: Language Models Can Teach Themselves to Use Tools"](https://arxiv.org/abs/2302.04761)** (Schick et al., 2023) — takes the fixed `web_search` tool above further: a model that learns for itself which API to call and when, with no hand-written system-message instruction to do so.